# Headless Dafne Thigh Segmentation — Asian MRI Dataset (Water, Lambda)

Runs the **Dafne Thigh model** on the `MRI_data_asian` Dixon WATER stacks — **Thigh only**.
Uses `dafne_dl.DynamicDLModel` directly (no GUI required).

Structure: `MRI_data_asian/MRI_data/{01-25}/Thigh/Water.nii.gz`  
Output: `~/dafne_asian_water_segs/{subject}/Thigh/Water_dafne_thigh.npz` (25 files)

## 1 — Upload to Lambda
```bash
# Asian MRI data
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/MRI_data_asian \
  ubuntu@<YOUR-LAMBDA-IP>:~/

# Dafne model file
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne_thigh_results/model_used/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_model/
```

## 2 — Download results when done
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_asian_water_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/asian_segs_water/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'dafne-dl', 'SimpleITK'])
print('Dependencies installed.')

In [ ]:
import glob
import os
import numpy as np
import SimpleITK as sitk
from dafne_dl import DynamicDLModel

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────

DATA_ROOT  = os.path.expanduser('~/MRI_data_asian/MRI_data')
OUTPUT_DIR = os.path.expanduser('~/dafne_asian_water_segs')

# Find the .model file — takes the first one found in ~/dafne_model/
_model_candidates = sorted(glob.glob(os.path.expanduser('~/dafne_model/*.model')))
if not _model_candidates:
    raise FileNotFoundError('No .model file found in ~/dafne_model/ — upload it first')
MODEL_PATH = _model_candidates[0]
print(f'Model : {MODEL_PATH}')

# Discover Thigh jobs
jobs = []
for subject in sorted(os.listdir(DATA_ROOT)):
    water_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'Water.nii.gz')
    if os.path.exists(water_path):
        jobs.append((subject, water_path))

print(f'Found {len(jobs)} Thigh Water stacks')
for subj, p in jobs:
    print(f'  {subj}  →  {p}')

In [ ]:
# ── Load Dafne model once ─────────────────────────────────────────────────────
model = DynamicDLModel.Load(open(MODEL_PATH, 'rb'))
print('Model loaded:', MODEL_PATH)

In [ ]:
# ── Run segmentation ──────────────────────────────────────────────────────────

for subject, water_path in jobs:
    out_subdir = os.path.join(OUTPUT_DIR, subject, 'Thigh')
    out_path   = os.path.join(out_subdir, 'Water_dafne_thigh.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {subject}/Thigh')
        continue

    print(f'\nProcessing: {subject}/Thigh')
    os.makedirs(out_subdir, exist_ok=True)

    img_sitk   = sitk.ReadImage(water_path)
    img_array  = sitk.GetArrayFromImage(img_sitk).astype(float)  # (D, H, W)
    spacing    = img_sitk.GetSpacing()                            # (x_mm, y_mm, z_mm)
    resolution = [spacing[0], spacing[1]]                        # 2D in-plane spacing
    print(f'  Shape: {img_array.shape}  Resolution: {resolution}')

    all_masks = {}  # {muscle_name: (D, H, W) uint8}

    for slice_idx in range(img_array.shape[0]):
        out = model({
            'image':            img_array[slice_idx],
            'resolution':       resolution,
            'split_laterality': True,
            'classification':   'Thigh',
        })
        for muscle_name, mask in out.items():
            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = np.asarray(mask, dtype=np.uint8)

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f'  slice {slice_idx + 1}/{img_array.shape[0]}')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}  |  muscles: {list(all_masks.keys())}')

print('\nAll done.')

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────────
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', 'Thigh', '*.npz')))
print(f'Output files found: {len(results)} / {len(jobs)}')
if results:
    sample = np.load(results[0])
    print(f'\nSample: {results[0]}')
    for name in sorted(sample.files):
        arr = sample[name]
        print(f'  {name}: shape={arr.shape}  voxels={int(arr.sum()):,}')